# Session 14 — The Agent Loop

**The concept:** an agent is a language model + tools + a loop + state.

**The lesson:** the loop is five lines and it fails in fifty ways. So it
needs three mandatory tripwires:

1. **Max iterations** — a hard cap. No infinite loops, ever.
2. **Cost ceiling** — a hard cap on spend.
3. **Step logging** — every step recorded. *If you cannot see the loop, you
   cannot debug it.*


In [ ]:
import json, os, sys, time, pathlib
sys.path.insert(0, os.path.join('..', 'src'))
print('python', sys.version.split()[0])


## 1. A normal run: tool call → result → answer


In [ ]:
from mock_llm import build_default_llm
from agent import Agent, AgentConfig
import tools as T

llm = build_default_llm()
agent = Agent(llm=llm, tools=T.registry, config=AgentConfig(max_iterations=8))
run = agent.run('What is the weather in Stockton?')

print('stop_reason:', run.stop_reason)
print('iterations :', run.iterations)
print('cost       : $%.8f' % run.total_cost)
print('answer     :', run.answer)


## 2. The log — this is what you show the class


In [ ]:
print(run.render_log())


## 3. Tripwire 1 — the iteration cap

Give the agent a script that never stops. It still terminates. Without this
cap the loop runs forever and burns real money in production.


In [ ]:
from mock_llm import MockLLM, ScriptedRule

looping = ScriptedRule(
    pattern=r'loop',
    content='Calling a tool again.',
    tool_calls=[{'id': 'c1', 'type': 'function',
                 'function': {'name': 'get_weather',
                              'arguments': '{"city": "Stockton, CA", "units": "f"}'}}],
)

loop_llm = MockLLM(script=[looping])
loop_agent = Agent(llm=loop_llm, tools=T.registry, config=AgentConfig(max_iterations=4))
loop_run = loop_agent.run('please loop forever')

print('stop_reason:', loop_run.stop_reason, '| iterations:', loop_run.iterations)
print('succeeded  :', loop_run.succeeded)


## 4. Tripwire 2 — the cost ceiling

The agent stops and *says so* rather than quietly burning money.


In [ ]:
pricey = build_default_llm(cost_per_1k_prompt_tokens=0.05, price_per_1k_completion_tokens=0.05)
pricey_agent = Agent(llm=pricey, tools=T.registry,
                     config=AgentConfig(max_iterations=50, max_cost_usd=0.0002))
pricey_run = pricey_agent.run('What is the weather in Stockton?')

print('stop_reason:', pricey_run.stop_reason)
print('answer     :', pricey_run.answer)


## 5. The teaching moment — the agent confidently lies

Give it an impossible task. It will claim success **without ever calling a
tool**. Nothing in the loop notices.

> This is why production agents need evals and verification.
> **"The agent said it was done" is not evidence.**


In [ ]:
liar_llm = build_default_llm()
liar = Agent(llm=liar_llm, tools=T.registry, config=AgentConfig(max_iterations=5))
liar_run = liar.run('Find the meaning of life using only the database tool')

called = [c['function']['name'] for s in liar_run.steps for c in s.tool_calls]
print('stop_reason :', liar_run.stop_reason)
print('tools called :', called, '  <-- none')
print('answer       :', liar_run.answer)
print()
print('The run "succeeded" as far as the loop is concerned -- and it is wrong.')


## Takeaway

- The agent loop is simple. The **tripwires** are the engineering.
- Cap iterations. Cap spend. Log every step.
- **An agent's claim of success is not evidence.** Verify it.
